环境：Apache Polaris (REST Catalog)-1.3.0 + Spark SQL-3.5.8 + Kyuubi-1.10.3 + MinIO

## 概念介绍
### 为什么需要表维护
想象一下，如果没有 Iceberg 的自动维护机制，你的数据湖会变成什么样？

+ **痛点一：小文件灾难**
    - 场景：你的程序每秒都在写入数据，每次只写几 KB。
    - 后果：一年后，你会有几百万个小文件。Spark 读取时，光是“打开文件”这个动作就要花几个小时，查询直接超时。
+ **痛点二：存储虚高**
    - 场景：你修改了一行数据，Iceberg 为了安全，不会直接改原文件，而是写个新文件，标记旧文件“作废”。
    - 后果：如果你不清理，那些“作废”的旧文件会永远留在 MinIO 里。
+ **痛点三：脏数据残留**
    - 场景：写入任务突然报错中断，或者集群宕机。
    - 后果：MinIO 里会留下一些没人认领的文件（孤儿文件）。它们既不是有效数据，也没被标记删除，就在那白白占用空间，还可能干扰统计。

Iceberg 的解决方案： Iceberg 提供了一套维护工具。它能在后台悄悄地把小文件合并成大文件、把过期的垃圾彻底删掉、把无人认领的脏文件清理掉。

### 核心维护任务
1. **合并小文件: **把无数个小碎片拼成几个大块头，提升查询速度。
2. **过期快照清理:** 删掉太久以前的历史版本，释放被占用的存储空间。
3. **孤儿文件清理: **扫描存储桶，删掉那些孤儿文件。
4. **排序优化: **把相关的数据 physically 放在一起，让查询更快。

---

## 实战案例
假设我们有一张高频写入的日志表 `dataspire_catalog.db_dev.app_logs`，现在它已经因为缺乏维护而变得臃肿。

### 第一步：诊断现状
先看看有多少小文件，以及历史快照堆积了多少。

```sql
-- 1. 创建表并模拟混乱状态
CREATE TABLE IF NOT EXISTS dataspire_catalog.db_dev.app_logs (
    log_id BIGINT,
    user_id STRING,
    action STRING,
    ts TIMESTAMP,
    dt STRING
) USING iceberg PARTITIONED BY (dt);

-- 模拟大量微小写入 (生产环境中这会产生数百个小文件)
INSERT INTO dataspire_catalog.db_dev.app_logs VALUES 
(1, 'u1', 'login', current_timestamp(), '2026-03-14');
-- ... 想象这里重复插入了 100 次 ...
INSERT INTO dataspire_catalog.db_dev.app_logs VALUES 
(100, 'u100', 'logout', current_timestamp(), '2026-03-14');

-- 2. 诊断：查看文件数量和平均大小
-- 痛点体现：文件数多，平均大小可能只有几 KB 或几 MB
SELECT 
    count(*) as file_count, 
    avg(file_size_in_bytes)/1024/1024 as avg_size_mb
FROM dataspire_catalog.db_dev.app_logs.files;

-- 3. 诊断：查看快照数量
SELECT count(*) as snapshot_count 
FROM dataspire_catalog.db_dev.app_logs.history;
```

### 第二步：合并小文件 
这是提升性能最关键的一步。我们将把小文件合并成标准大小（如 128MB）。

```sql
-- 执行合并
CALL dataspire_catalog.system.rewrite_data_files(
    table => 'db_dev.app_logs',
    options => map(
        'target-file-size-bytes', '134217728',  -- 目标：128 MB
        'min-input-files', '5',                -- 至少凑够 5 个文件才合并，避免浪费资源
        'max-concurrent-file-group-rewrites', '10' -- 并行度
    )
);

-- 验证：文件变少了，单个文件变大了
SELECT 
    count(*) as new_file_count, 
    avg(file_size_in_bytes)/1024/1024 as new_avg_size_mb
FROM dataspire_catalog.db_dev.app_logs.files;
```

注意：此时旧的小文件在 MinIO 里还在，只是被标记为“无效”。

### 第三步：清理过期快照
现在我们要真正删除那些“无效”的旧文件，释放 MinIO 空间。

```sql
-- 清理策略：保留最近 3 天 或 最近 10 个快照
CALL dataspire_catalog.system.expire_snapshots(
    table => 'db_dev.app_logs',
    retain_last => 10,             -- 安全线：至少留 10 个
    older_than => interval '3 days' -- 时间线：3 天前的都删掉
);
```

### 第四步：清理孤儿文件
最后，扫荡一遍 MinIO，删掉所有没被引用的文件。

```sql
-- 清理策略：只删那些 1 天前就存在的孤儿文件 (防止误删正在写的文件)
CALL dataspire_catalog.system.delete_orphan_files(
    table => 'db_dev.app_logs',
    older_than => interval '1 days' 
);

-- 最终验证：存储空间应该显著下降
SELECT sum(file_size_in_bytes)/1024/1024 as total_size_mb 
FROM dataspire_catalog.db_dev.app_logs.files;
```

### 进阶：排序优化 
如果经常查 `user_id`，可以让数据按用户物理排序。

```sql
CALL dataspire_catalog.system.rewrite_data_files(
    table => 'db_dev.app_logs',
    strategy => 'sort',
    options => map(
        'target-file-size-bytes', '134217728',
        'sort-order', 'user_id' 
    )
);
```

---

## 最佳实践
### 自动化调度建议
不要手动跑，设好定时任务：

| 任务 | 频率 | 关键参数建议 | 目的 |
| --- | --- | --- | --- |
| 合并小文件 | 每小时 | `min-input-files`: 5   `target-size`: 128MB | 保持查询飞快 |
| 清理旧快照 | 每天凌晨 | `retain_last`: 10   `older_than`: 3 Days | 省钱 (存储费) |
| 清理孤儿 | 每周一次 | `older_than`: 3 Days (一定要保守!) | 扫除脏数据 |


### 三条保命红线
1. 孤儿文件时间窗口 (`older_than`): 
    - 绝对不要设置得太短（比如 1 小时）。如果有长运行任务还没写完，会被误删导致数据丢失。建议至少 24-72 小时。
2. 快照保留数量 (`retain_last`): 
    - 即使设置了时间，也要保留一定数量（如 10 个），防止时间设置太短把刚合并好的新快照也删了。
3. 资源隔离: 
    - 维护任务很吃 CPU 和 IO。最好在业务低峰期跑，或者用独立的资源队列，别跟正常 ETL 抢资源。

---

## 总结速查表
| 痛点 | 解决方案 (Spark SQL) | 核心作用 |
| --- | --- | --- |
| 文件太小，查询慢 | `CALL system.rewrite_data_files(...)` | 提速：小变大 |
| 历史太多，存储贵 | `CALL system.expire_snapshots(...)` | 省钱：删旧版本 |
| 故障残留，有脏数据 | `CALL system.delete_orphan_files(...)` | 洁净：删黑户 |
| 查询扫描范围大 | `CALL system.rewrite_data_files(..., strategy=>'sort')` | 加速：数据排序 |


